# Load Libraries

In [ ]:
pip install -qqq ragas==0.4.3 langchain-community==0.4.1 langchain_huggingface PyPDF2 faiss-cpu evaluate langchain-text-splitters rank-bm25 chromadb wtpsplit rapidfuzz sentence_transformers datasets langchain_openai

In [ ]:
from datasets import load_dataset
from huggingface_hub import login, snapshot_download, InferenceClient
from google.colab import userdata
import pandas as pd
import numpy as np
import os
from pathlib import Path
import difflib
import asyncio
from rich.progress import track

import PyPDF2
import requests
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import re
from evaluate import load
from rank_bm25 import BM25Okapi
import chromadb
import unicodedata
from torch.utils.data import DataLoader
from wtpsplit import SaT
from rapidfuzz import fuzz
import ast
from transformers import AutoTokenizer, pipeline

from openai import OpenAI, AsyncOpenAI
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerCorrectness
from datasets import Dataset
from dataclasses import dataclass
from typing import Optional
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.llms import llm_factory

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%cd /content/drive/MyDrive/RAG
cwd = os.getcwd()

# Load Dataset

In [ ]:
# snapshot_download(
#         repo_id='theatticusproject/cuad',
#         repo_type="dataset",
#         local_dir=f'{cwd}/data',
#         ignore_patterns=["*.gitattributes", ".gitignore"],
#         token = hf_token
#     )

In [ ]:
def pdf_files_paths_list(pdf_contracts_dir):
    pdf_files_paths = []
    for folder in Path(pdf_contracts_dir).iterdir():
        base = folder / os.listdir(folder)[0] if folder.name == 'Part_II' else folder
        for sub_folder in base.iterdir():
            pdf_files_paths.extend(sub_folder / pdf for pdf in os.listdir(sub_folder))
    return pdf_files_paths

In [ ]:
def txt_files_paths_list(txt_contracts_dir):
    txt_files_paths = []
    for folder in Path(txt_contracts_dir).iterdir():
        for txt_file in folder.iterdir():
            txt_files_paths.append(txt_file)
    return txt_files_paths

In [ ]:
data_folder = f'{cwd}/data/CUAD_v1'
pdf_contracts = f'{data_folder}/full_contract_pdf'
txt_contracts = f'{data_folder}/full_contract_txt'

In [ ]:
pdf_files_paths = pdf_files_paths_list(pdf_contracts)
assert len(pdf_files_paths) == 510

In [ ]:
txt_files_paths = txt_files_paths_list(txt_contracts)
len(txt_files_paths)

In [ ]:
master_clauses = pd.read_csv(f'{data_folder}/master_clauses.csv')
master_clauses.head()

In [ ]:
master_clauses.rename(columns={'Notice Period To Terminate Renewal- Answer': 'Notice Period To Terminate Renewal-Answer'}, inplace=True)

In [ ]:
master_clauses.fillna('', inplace=True)

In [ ]:
# 1. Filter only the columns that end with '-Answer'
# answer_cols = [col for col in master_clauses.columns if col.endswith("-Answer")]
answer_cols = [col for col in master_clauses.columns if col.endswith("-Answer")]

# 2. Clean the target columns (convert to string, treat empty strings as NaN)
cleaned_df = master_clauses[answer_cols].astype(str).replace({"": np.nan, "None": np.nan})

# 3. Find the string lengths for all elements
lengths = cleaned_df.applymap(lambda x: len(x) if pd.notnull(x) else 0)

# 4. Locate the maximum length and pull the longest string
if not lengths.empty and lengths.max().max() > 0:
    # Find the column and row index of the max value
    max_col = lengths.max().idxmax()
    max_idx = lengths[max_col].idxmax()

    longest_string = master_clauses.loc[max_idx, max_col]
    print(f"Longest string found in column '{max_col}' at index {max_idx}:")
    print(f"Length: {len(longest_string)} characters")
    print(f"Content: {longest_string}")
else:
    print("No valid strings found in the '-Answer' columns.")

In [ ]:
len(longest_string.split(' '))

In [ ]:
master_clauses.shape

In [ ]:
label_groups = ['Filename']
folder = 'label_group_xlsx'
for f in os.listdir(f'{data_folder}/{folder}'):
    if f == 'Label Report - Uncapped Liability (Group 5).xlsx':
        label_groups.append('Uncapped Liability')
    else:
        df = pd.read_excel(f'{data_folder}/{folder}/{f}')
        labels = list(df.columns.values[1:])
        for label in labels:
            if 'Answer' in label:
              label_groups.pop()
            label_groups.append(label)
        # print(label, f)

In [ ]:
assert len(label_groups) == 41

In [ ]:
clauses = [clause.lower() for clause in master_clauses.columns.values]
for label in label_groups:
    if label.lower() not in clauses:
        match_str = difflib.get_close_matches(label.lower(), clauses, n=1)
        print(match_str, label.lower())

# RAG Pipeline

#### Loading Language Models

In [ ]:
os.environ["HF_TOKEN"] = userdata.get('hf_token')

In [ ]:
%%capture
text_splitter = SaT("sat-3l")
embedding_model_name = 'BAAI/bge-large-en-v1.5'
embedding_model = SentenceTransformer(embedding_model_name)
cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)

# chat_model = "Qwen/Qwen2.5-7B-Instruct"
chat_model = "qwen/qwen-2.5-7b-instruct"
llm_judge = "qwen/qwen-2.5-72b-instruct"
# pipe = pipeline(
#     "text-generation",
#     model=chat_model,
#     device_map="auto",
#     max_new_tokens=50,
#     do_sample=False,
# )

free_embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

#### Loading Vector Database, Query templates, API tokens

In [ ]:
embed_client = InferenceClient()
client = chromadb.PersistentClient(path="./cuad_vector_db_copy")
pdf_collection = client.get_collection(name = 'atticus_vector_data_cp')
query_collection = client.get_collection(name = 'query_collection')

In [ ]:
os.environ['OPENAI_API_KEY'] = userdata.get('openai')
os.environ['OPENROUTER_API_KEY'] = userdata.get('openrouter')

In [ ]:
def template_chunk_retrieval(user_query, doc_collection):
    query_embedding = embed_client.feature_extraction(
        text=[user_query],
        model=embedding_model_name)

    retrieved_column = doc_collection.query(query_embeddings=query_embedding, n_results=1)['metadatas'][0][0]['source']
    print(retrieved_column)
    template_chunk = search_templates_dict[retrieved_column]
    template_embedding = embed_client.feature_extraction(
        text=[template_chunk],
        model=embedding_model_name)

    return retrieved_column, template_chunk, template_embedding

In [ ]:
search_templates_df = pd.read_csv('search_templates.csv')

In [ ]:
search_templates_dict = search_templates_df.set_index('Column')['Query'].to_dict()

### Creating Vector Embeddings for Single Document

In [ ]:
def extract_text_from_pdf(file_path):
    with open(file_path, 'rb') as file:
      reader = PyPDF2.PdfReader(file)
      text = ""
      for page in reader.pages:
          page_text = page.extract_text()
          if page_text:
              text += page_text + "\n"
    text = unicodedata.normalize("NFKC", text)
    return text

In [ ]:
def extract_chunks(paragraphs):
    chunks = []
    for paragraph in paragraphs:
        chunk = ''
        for sentence in paragraph:
            chunk += sentence + ' '
        chunks.append(chunk)

    return chunks

In [ ]:
# file_path = f'{cwd}/CreditcardscomInc_20070810_S-1_EX-10.33_362297_EX-10.33_Affiliate Agreement.pdf'
file_path = f'{cwd}/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf'
# file_path = '/content/drive/MyDrive/RAG/data/CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/UsioInc_20040428_SB-2_EX-10.11_1723988_EX-10.11_Affiliate Agreement 2.pdf'
# file_path = '/content/drive/MyDrive/RAG/data/CUAD_v1/full_contract_pdf/Part_I/Co_Branding/MphaseTechnologiesInc_20030911_10-K_EX-10.15_1560667_EX-10.15_Co-Branding Agreement.pdf'

In [ ]:
text = extract_text_from_pdf(file_path)

In [ ]:
paragraphs = text_splitter.split(text, do_paragraph_segmentation=True)

In [ ]:
chunks = extract_chunks(paragraphs)

### Creating Vector Database

#### Documents

In [ ]:
embed_client = InferenceClient()

In [ ]:
# Initialize the client (persists data to a local directory)
client = chromadb.PersistentClient(path="./cuad_vector_db_copy")
collection = client.get_collection(name = 'atticus_vector_data_cp')

In [ ]:
n_start = 189
n_end = n_start + 1

for file_path in track(pdf_files_paths[n_start: n_end], 'Processing...'):
    # Create a collection (similar to a table in SQL)
    name = Path(file_path).stem.replace(" ", "-") # Replace spaces with hyphens for valid collection name

    text = extract_text_from_pdf(file_path)
    paragraphs = text_splitter.split(text, do_paragraph_segmentation=True)

    chunks = extract_chunks(paragraphs)
    # embeddings = embed_client.feature_extraction(
    # text=chunks,
    # model=embedding_model_name)
    embeddings = embedding_model.encode(chunks, convert_to_numpy=True)

    ids = [name + 'chunk_' + str(id) for id in range(len(chunks))]
    metadatas = [{'source': name}]*len(chunks)

    collection.add(
        ids = ids,
        documents = chunks,
        embeddings = embeddings,
        metadatas = metadatas
    )

#### Queries

In [ ]:
queries = {
    'What is the agreement date of the contract': 'Agreement Date',
    'On what date is the contract effective?': 'Effective Date',
    'On what date will the contract’s initial term expire?': 'Expiration Date',
    'What is the renewal term after the initial term expires? This includes automatic extensions and unilateral extensions with prior notice.': 'Renewal Term',
    'What is the notice period required to terminate renewal?': 'Notice Period To Terminate Renewal',
    'Which state/country’s law governs the interpretation of the contract?': 'Governing Law',
    'Is there a clause that if a third party gets better terms on the licensing or sale of technology/goods/services described in the contract, the buyer of such technology/goods/services under the contract shall be entitled to those better terms?': 'Most Favored Nation',
    'Is there a restriction on the ability of a party to compete with\nthe counterparty or operate in a certain geography or business or\ntechnology sector?': 'Non-Compete',
    'Is there an exclusive dealing commitment with the counterparty?\nThis includes a commitment to procure all “requirements” from one\nparty of certain technology, goods, or services or a prohibition on\nlicensing or selling technology, goods or services to third parties,\nor a prohibition on collaborating or working with other parties),\nwhether during the contract or after the contract ends (or both).\n': 'Exclusivity',
    'Is a party restricted from contracting or soliciting customers or\npartners of the counterparty, whether during the contract or after\nthe contract ends (or both)?\n': 'No-Solicit Of Customers',
    'This category includes the exceptions or carveouts to Non-Compete, Exclusivity and No-Solicit of Customers above.': 'Competitive Restriction Exception',

    'Is there a restriction on a party’s soliciting or hiring employees\nand/or contractors from the counterparty, whether during the contract or after the contract ends (or both)?': 'No-Solicit Of Employees',
    'Is there a requirement on a party not to disparage the counterparty?': 'Non-Disparagement',
    'Can a party terminate this contract without cause (solely by giving a notice and allowing a waiting period to expire)?': 'Termination For Convenience',
    'Is there a clause granting one party a right of first refusal, right of\nfirst offer or right of first negotiation to purchase, license, market, or\ndistribute equity interest, technology, assets, products or services?': 'Rofr/Rofo/Rofn',

    'Does one party have the right to terminate or is consent or notice\nrequired of the counterparty if such party undergoes a change of\ncontrol, such as a merger, stock sale, transfer of all or substantially\nall of its assets or business, or assignment by operation of law?': 'Change Of Control',
    'Is consent or notice required of a party if the contract is assigned to a third party?': 'Anti-Assignment',
    'Is one party required to share revenue or profit with the counterparty\nfor any technology, goods, or services?\n': 'Revenue/Profit Sharing',
    'Is there a restriction on the ability of a party to raise or reduce prices\nof technology, goods, or services provided?\n': 'Price Restrictions',
    'Is there a minimum order size or minimum amount or units pertime\nperiod that one party must buy from the counterparty under\nthe contract?\n': 'Minimum Commitment',
    'Is there a fee increase or consent requirement, etc. if one party’s\nuse of the product/services exceeds certain threshold?\n': 'Volume Restriction',
    'Does intellectual property created by one party become the property\nof the counterparty, either per the terms of the contract or upon the\noccurrence of certain events?\n': 'Ip Ownership Assignment',

    'Is there any clause providing for\njoint or shared ownership of intellectual property between the parties to the contract?\n': 'Joint Ip Ownership',
    'Does the contract contain a license granted by one party to its counterparty?': 'License Grant',

    'Does the contract limit the ability of a party to transfer the license\nbeing granted to a third party?\n': 'Non-Transferable License',

    'Does the contract contain a license grant by affiliates of the licensor\nor that includes intellectual property of affiliates of the licensor?\n': 'Affiliate License-Licensor',

    'Does the contract contain a license grant to a licensee (incl. sublicensor)\nand the affiliates of such licensee/sublicensor?\n': 'Affiliate License-Licensee',
    'Is there a clause granting one party an “enterprise,” “all you can eat”\nor unlimited usage license?\n': 'Unlimited/All-You-Can-Eat-License',
    'Does the contract contain a license grant that is irrevocable or perpetual?': 'Irrevocable Or Perpetual License',
    'Is one party required to deposit its source code into escrow with\na third party, which can be released to the counterparty upon the\noccurrence of certain events (bankruptcy, insolvency, etc.)?': 'Source Code Escrow',
    'Is a party subject to obligations after the termination or expiration\nof a contract, including any post-termination transition, payment,\ntransfer of IP, wind-down, last-buy, or similar commitments?\n': 'Post-Termination Services',
    'Does a party have the right to audit the books, records, or\nphysical locations of the counterparty to ensure compliance with the\ncontract?': 'Audit Rights',
    'Is a party’s liability uncapped upon the breach of its obligation\nin the contract? This also includes uncap liability for a particular\ntype of breach such as IP infringement or breach of confidentiality\nobglication.': 'Uncapped Liability',
    'Does the contract include a cap on liability upon the breach of a party’s obligation? \nThis includes time limitation for the counterparty to bring claims or maximum amount for recovery.': 'Cap On Liability',

    'Does the contract contain a clause that would award either party\nliquidated damages for breach or a fee upon the termination of a\ncontract (termination fee)?\n': 'Liquidated Damages',
    'What is the duration of any warranty against defects or errors in\ntechnology, products, or services provided under the contract?': 'Warranty Duration',
    'Is there a requirement for insurance that must be maintained by one\nparty for the benefit of the counterparty?': 'Insurance',
    'Is a party restricted from CONTESTING the validity of the counterparty’s ownership of intellectual property or otherwise bringing a claim\nagainst the counterparty for matters unrelated to the contract?': 'Covenant Not To Sue',
    'Is there a non-contracting party who is a beneficiary to some or all\nof the clauses in the contract and therefore can enforce its rights\nagainst a contracting party?': 'Third Party Beneficiary',
    'Which parties/corporations are subject to the Agreement?': 'Parties'
}

In [ ]:
client = chromadb.PersistentClient(path="./cuad_vector_db_copy")
new_collection = client.get_or_create_collection(name="query_collection")

In [ ]:
for query, column in queries.items():
    embeddings = embed_client.feature_extraction(
    text=[query],
    model=embedding_model_name)

    metadatas = [{'source': column}]
    ids = [f'{column}_1']

    new_collection.add(
        ids = ids,
        documents = query,
        embeddings = embeddings,
        metadatas = metadatas
)
    # print('done with ', column)

## Hybrid Retrieval

In [ ]:
def hybrid_search(query, query_embedding, collection, name, use_cross_enc = True, top_k=5):
    # 1. Dense retrieval
    dense_results = collection.query(query_embeddings=[query_embedding], n_results=10,
                                     where = {'source': name})
    dense_ids = dense_results["ids"][0]  # ["chunk_0", "chunk_3", ...]

    # 2. Sparse retrieval (BM25)
    vector_data = collection.get(where={"source": name}, include = ['documents'])
    chunks = vector_data["documents"]

    tokenized_chunks = [tokenizer.tokenize(chunk) for chunk in chunks]
    bm25 = BM25Okapi(tokenized_chunks)
    tokenized_query = tokenizer.tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    top_sparse_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:20]
    sparse_ids = [f"{name}chunk_{i}" for i in top_sparse_indices]  # convert to chunk IDs

    # 3. Merge with RRF
    def reciprocal_rank_fusion(dense_ids, sparse_ids, k=60):
        scores = {}
        for rank, doc_id in enumerate(dense_ids):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        for rank, doc_id in enumerate(sparse_ids):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)

    if use_cross_enc:
        candidate_ids = reciprocal_rank_fusion(dense_ids, sparse_ids)[:10]

        candidate_data = collection.get(ids=candidate_ids)
        candidate_texts = candidate_data["documents"]

        # # 4. CROSS-ENCODER RERANKING (Precision Stage)
        # # Pair the query with each candidate text for scoring
        rerank_pairs = [[query, text] for text in candidate_texts]
        scores = cross_encoder_model.predict(rerank_pairs)

        # 5. Pair scores with documents and sort
        doc_score_pairs = list(zip(candidate_texts, scores))
        doc_score_pairs.sort(key=lambda x: x[1], reverse=True)
        # Return only the top_k requested results
        return [text for text, score in doc_score_pairs[:top_k]]
    else:
        final_ids = reciprocal_rank_fusion(dense_ids, sparse_ids)[:top_k]
        final_texts = [collection.get(ids = id)['documents'][0] for id in final_ids]

        return  final_texts

In [ ]:
# query_dict = {}
hit_rate_dict = {}
mrr_dict = {}

In [ ]:
queries = pd.read_csv('query.csv')
query_dict = queries.set_index('Column')['Query'].to_dict()

In [ ]:
def normalize(text: str) -> str:
    """Lowercase and strip punctuation for comparison."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text)       # collapse spaces
    return text.strip()

In [ ]:
def is_phrase_in_text(phrase: str, text: str, threshold: int = 80) -> dict:
    norm_text   = normalize(text)
    norm_phrase = normalize(phrase)

    # partial_ratio checks if phrase is contained as a substring
    score = fuzz.partial_ratio(norm_phrase, norm_text)

    return {
        "contained": score >= threshold,
        "score": score,
        "verdict": "YES" if score >= threshold else "NO"
    }

#### Creating Template Chunks for Columns

In [ ]:
context_cols = [col for col in master_clauses.columns if not col.endswith("-Answer")]
assert len(context_cols) == 43

In [ ]:
rem_cols = [col for col in context_cols if col not in list(query_dict.keys())]

In [ ]:
# column = 'Parties'
# column = 'Agreement Date'
# column = 'Governing Law'
# column = 'Termination For Convenience'
# column = 'License Grant'
# column = 'Non-Transferable License'
# column = 'Cap On Liability'
# column = 'Effective Date'
# column = 'Expiration Date'
# column = 'Renewal Term'
# column = 'Notice Period To Terminate Renewal'
# column = 'Most Favored Nation'
# column = 'Competitive Restriction Exception'
# column = 'Non-Compete'
# column = 'Exclusivity'
# column = 'No-Solicit Of Customers'
# column = 'No-Solicit Of Employees'
# column = 'Non-Disparagement'

In [ ]:
len(context_cols)

In [ ]:
column = rem_cols[25]
column

In [ ]:
for pdf_file in pdf_files_paths[:200]:
    pdf_filename = pdf_file.name
    try:
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
    except:
        pdf_filename = Path(pdf_filename).stem + '.PDF'
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
    answers = ast.literal_eval(answers)
    if answers == []:
        continue
    else:
        break
answers[0]

In [ ]:
query = answers[0]

In [ ]:
# query = """THIS NETWORK AFFILIATE AGREEMENT (this “Agreement”) is made as of this 14th day of March, 2011
# by and between National CineMedia,
# LLC, a Delaware limited liability company (“NCM”), and Digital Cinema Destinations Corp.,
# a Delaware corporation (“Network Affiliate” and with
# NCM, each a “Party” and collectively, the “Parties”)."""
# query = f'{column}: Which state/country’s law governs the interpretation of the contract?'
# query = f"{column}: Can a party terminate this contract without cause (solely by giving a notice and allowing a waiting period to expire)?"
# query = f"{column}: Does the contract contain a license granted by one party to its counterparty?"
# query = f"{column}: Does the contract include a cap on liability upon the breach of a party’s obligation? This includes time limitation for the counterparty to bring claims or maximum amount for recovery."
# query = f'{column}: On what date is the contract is effective?'
# query = f'{column}: On what date will the contract’s initial term expire?'
# query = f'{column}: What is the renewal term after the initial term expires? This includes automatic extensions and unilateral extensions with prior notice.'
# query = f'{column}: What is the notice period required to terminate renewal?'
# query = f"""{column}: Is there a clause that if a third party gets better terms on the licensing
# or sale of technology/goods/services described in the contract, the
# buyer of such technology/goods/services under the contract shall
# be entitled to those better terms?
# """
# query = f"""{column}: This category includes the exceptions or carveouts to Non-Compete,
#  Exclusivity and No-Solicit of Customers above."""
# query = f"""{column}: Is there a restriction on the ability of a party to compete with
# the counterparty or operate in a certain geography or business or
# technology sector?"""
# query = f"""{column}: Is there an exclusive dealing commitment with the counterparty?
# This includes a commitment to procure all “requirements” from one
# party of certain technology, goods, or services or a prohibition on
# licensing or selling technology, goods or services to third parties,
# or a prohibition on collaborating or working with other parties),
# whether during the contract or after the contract ends (or both)."""
# query = f"""{column}: Is a party restricted from contracting or soliciting customers or
# partners of the counterparty, whether during the contract or after
# the contract ends (or both)?
# """
# query = f"""{column}: Is there a restriction on a party’s soliciting or hiring employees
# and/or contractors from the counterparty, whether during the contract or after the contract ends (or both)?"""
# query = f"""{column}: Is there a requirement on a party not to disparage the counterparty?"""
# query = f"""{column}: Is there a clause granting one party a right of first refusal, right of
# first offer or right of first negotiation to purchase, license, market, or
# distribute equity interest, technology, assets, products or services?"""
# query = f"""
# {column}: Does one party have the right to terminate or is consent or notice
# required of the counterparty if such party undergoes a change of
# control, such as a merger, stock sale, transfer of all or substantially
# all of its assets or business, or assignment by operation of law?
# """
# query = f'{column}: Is consent or notice required of a party if the contract is assigned to a third party?'
# query = f"""{column}: Is one party required to share revenue or profit with the counterparty
# for any technology, goods, or services?
# """
# query = f"""{column}:Is there a restriction on the ability of a party to raise or reduce prices
# of technology, goods, or services provided?
# """
# query = f"""{column}: Is there a minimum order size or minimum amount or units pertime
# period that one party must buy from the counterparty under
# the contract?
# """
# query = f"""{column}: Is there a fee increase or consent requirement, etc. if one party’s
# use of the product/services exceeds certain threshold?
# """
# query = f"""{column}:Does intellectual property created by one party become the property
# of the counterparty, either per the terms of the contract or upon the
# occurrence of certain events?
# """
# query = f"""{column}: Is there any clause providing for
# joint or shared ownership of intellectual property between the parties to the contract?"""
# query = f"""{column}: Does the contract contain a license grant by affiliates of the licensor
# or that includes intellectual property of affiliates of the licensor?
# """
# query = f""" {column}: Does the contract contain a license grant to a licensee (incl. sublicensor)
# and the affiliates of such licensee/sublicensor?
# """
# query = f"""{column}: Is there a clause granting one party an “enterprise,” “all you can eat”
# or unlimited usage license?
# """
# query = f"""{column}: Does the contract contain a license grant that is irrevocable or
# perpetual?
# """
# query = f"""{column}: Is one party required to deposit its source code into escrow with
# a third party, which can be released to the counterparty upon the
# occurrence of certain events (bankruptcy, insolvency, etc.)?"""
# query = f"""{column}: Is a party subject to obligations after the termination or expiration
# of a contract, including any post-termination transition, payment,
# transfer of IP, wind-down, last-buy, or similar commitments?
# """
# query = f"""{column}: Does a party have the right to audit the books, records, or
# physical locations of the counterparty to ensure compliance with the
# contract?"""
# query = f"""{column}: Is a party’s liability uncapped upon the breach of its obligation
# in the contract? This also includes uncap liability for a particular
# type of breach such as IP infringement or breach of confidentiality
# obligation."""
# query = f"""{column}: Does the contract contain a clause that would award either party
# liquidated damages for breach or a fee upon the termination of a
# contract (termination fee)?
# """
# query = f"""{column}: What is the duration of any warranty against defects or errors in
# technology, products, or services provided under the contract?"""
# query = f"""{column}: Is there a requirement for insurance that must be maintained by one
# party for the benefit of the counterparty?"""
# query = f"""{column}: Is a party restricted from contesting the validity of the counterparty’s
# ownership of intellectual property or otherwise bringing a claim
# against the counterparty for matters unrelated to the contract?"""
query = f"""{column}: Is there a non-contracting party who is a beneficiary to some or all
of the clauses in the contract and therefore can enforce its rights
against a contracting party?"""

# query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]
# query = query_dict[column]

In [ ]:
print(column)
print(query)
query_embedding = embed_client.feature_extraction(
    text=[query],
    model=embedding_model_name,
    )
query_dict[column] = query

In [ ]:
len(query_dict.keys())

### Validation

In [ ]:
ndocs_dict = {}

In [ ]:
for column in track(hit_rate_df['Column'].values, description="Processing..."):
    n_docs = 0
    for pdf_file in pdf_files_paths[:200]:
        pdf_filename = pdf_file.name
        name = Path(pdf_file).stem.replace(" ", "-")
        try:
            answers = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
        except:
            pdf_filename = Path(pdf_filename).stem + '.PDF'
            answers = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
        answers = ast.literal_eval(answers)
        if not answers == []:
            n_docs += 1
    ndocs_dict[column] = n_docs

In [ ]:
val_chunks_no_enc= {}
mmr = {}
n_docs = 0

for pdf_file in track(pdf_files_paths[:200], description="Processing..."):
    pdf_filename = pdf_file.name
    name = Path(pdf_file).stem.replace(" ", "-")

    retrieved_chunks = hybrid_search(query, query_embedding[0], collection, name, use_cross_enc = False, top_k = 5)
    try:
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
    except:
        pdf_filename = Path(pdf_filename).stem + '.PDF'
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
    answers = ast.literal_eval(answers)
    if not answers == []:
        n_docs += 1
        mmr[pdf_filename] = 0.
        for ind, chunk in enumerate(retrieved_chunks):
            result = []
            for answer in answers:
                result.append(is_phrase_in_text(answer, chunk)['contained'])
            if all(result) and mmr[pdf_filename] == 0.:
                # print(pdf_filename, ind)
                val_chunks_no_enc[pdf_filename] = chunk
                mmr[pdf_filename] = 1./(ind + 1)
    # print('done with file:', pdf_filename)

In [ ]:
print('column:', column)
print('num docs:', n_docs)
print('Hit rate@5:', np.round(len(val_chunks_no_enc.keys())/n_docs, 2))
print('Mean Reciprocal Rank:', np.round(np.mean(list(mmr.values())), 2))

In [ ]:
hit_rate_dict[column] = len(val_chunks_no_enc.keys())/n_docs
mrr_dict[column] = np.mean(list(mmr.values()))

In [ ]:
# query_df = pd.DataFrame(query_dict.items(), columns=['Column', 'Query'])
# hit_rate_df = pd.DataFrame(hit_rate_dict.items(), columns=['Column', 'Hit Rate'])
# mrr_df = pd.DataFrame(mrr_dict.items(), columns=['Column', 'Mean Reciprocal Rank'])

# query_df.to_csv('query.csv', index=False)
# hit_rate_df.loc[hit_rate_df['Column'] == column, 'Hit Rate'] = hit_rate_dict[column]
# mrr_df.loc[mrr_df['Column'] == column, 'Mean Reciprocal Rank'] = mrr_dict[column]

hit_rate_df.loc[len(hit_rate_df)] = [column, hit_rate_dict[column]]
mrr_df.loc[len(mrr_df)] = [column, mrr_dict[column]]

In [ ]:
hit_rate_df['n_docs'] = hit_rate_df['Column'].map(ndocs_dict)

In [ ]:
hit_rate_df.tail()

In [ ]:
# mrr_df['n_docs'] = mrr_df['Column'].map(ndocs_dict)
mrr_df.tail()

In [ ]:
print('average hit rate:', np.round(np.mean(hit_rate_df['Hit Rate'].values), 2))
print('average mrr:', np.round(np.mean(mrr_df['Mean Reciprocal Rank'].values), 2))

In [ ]:
weighted_hit_rate_mean = ((hit_rate_df['Hit Rate']*hit_rate_df['n_docs'])/np.sum(hit_rate_df['n_docs'])).sum()
weighted_mrr_mean = ((mrr_df['Mean Reciprocal Rank']*mrr_df['n_docs'])/np.sum(mrr_df['n_docs'])).sum()

In [ ]:
print('weighted average hit rate:', np.round(weighted_hit_rate_mean, 2))
print('weighted average mrr:', np.round(weighted_mrr_mean, 2))

In [ ]:
query_df = pd.DataFrame(query_dict.items(), columns=['Column', 'Query'])
query_df.to_csv('query.csv', index=False)
query_df.tail()

In [ ]:
hit_rate_df.to_csv('hit_rate.csv', index=False)
mrr_df.to_csv('mrr.csv', index=False)

In [ ]:
hit_rate_df = pd.read_csv('hit_rate.csv')
mrr_df = pd.read_csv('mrr.csv')

# Answer Generation

In [ ]:
os.environ['OPENAI_API_KEY'] = userdata.get('openai')
os.environ['OPENROUTER_API_KEY'] = userdata.get('openrouter')

In [ ]:
chat_client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ.get('OPENROUTER_API_KEY')
)

In [ ]:
def build_prompt(query: str, chunks: list) -> list[dict]:
    # Handle both plain strings and RetrievedChunk objects

    if not chunks:
        return [
            {"role": "user", "content": f"No relevant context was found."}
        ]

    context = chunks[0]
    messages = [
    {"role": "system", "content": """You are an assistant that extracts specific facts from legal text.
    Answer the question based only on the provided context.
    If the information is not explicitly or implicitly mentioned, return NA.
    Otherwise, be concise and use the fewest words to answer.
        """},
    {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"}
    ]
    return messages

In [ ]:
def template_chunk_retrieval(user_query, doc_collection):
    query_embedding = embed_client.feature_extraction(
        text=[user_query],
        model=embedding_model_name)

    retrieved_column = doc_collection.query(query_embeddings=query_embedding, n_results=1)['metadatas'][0][0]['source']
    print(retrieved_column)
    template_chunk = search_templates_dict[retrieved_column]
    template_embedding = embed_client.feature_extraction(
        text=[template_chunk],
        model=embedding_model_name)

    return retrieved_column, template_chunk, template_embedding

In [ ]:
search_templates_df = pd.read_csv('search_templates.csv')
search_templates_df.head()

In [ ]:
search_templates_dict = search_templates_df.set_index('Column')['Query'].to_dict()

In [ ]:
query_collection = client.get_or_create_collection(name="query_collection")

In [ ]:
user_query = """Does the contract grant enforcement rights or benefits to any non-contracting third party?
"""

retrieved_column, template_chunk, template_embedding = template_chunk_retrieval(user_query, query_collection)

print('Column:', retrieved_column)
print('Template chunk:', template_chunk)

In [ ]:
llm_responses = {}
answers_dict = {}

for pdf_file in pdf_files_paths[:200]:
    pdf_filename = pdf_file.name
    name = Path(pdf_file).stem.replace(" ", "-")

    retrieved_chunks = hybrid_search(template_chunk, template_embedding[0], collection, name, use_cross_enc = False, top_k = 1)
    # retrieved_chunk_dict[pdf_filename] = retrieved_chunks[0]
    try:
        # correct_chunk = master_clauses[master_clauses['Filename'] == pdf_filename][retrieved_column].values[0]
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][retrieved_column+'-Answer'].values[0]
    except:
        pdf_filename = Path(pdf_filename).stem + '.PDF'
        # correct_chunk = master_clauses[master_clauses['Filename'] == pdf_filename][retrieved_column].values[0]
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][retrieved_column+'-Answer'].values[0]
    # correct_chunk = ast.literal_eval(correct_chunk)

    answers_dict[pdf_filename] = answers

    messages = build_prompt(
        query          = user_query,
        chunks         = retrieved_chunks,
    )

    response = chat_client.chat.completions.create(
      model=chat_model,
      messages=messages,
      max_tokens=400,
      temperature=0.1
    )

    # response = hf_client.chat.completions.create(
    #     model=chat_model,
    #     messages=messages,
    #     max_tokens=400,
    #     temperature=0.1
    # )

    llm_answer = response.choices[0].message.content
    print(llm_answer)
    print('ground truth:', answers)
    print('\n')
    llm_responses[pdf_filename] = llm_answer

In [ ]:
try:
    try:
        llm_responses_df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'],inplace=True, errors='ignore')
    except: pass
    llm_responses_df[f'{retrieved_column}-LLM'] = llm_responses_df['Filename'].map(llm_responses)
    llm_responses_df[f'{retrieved_column}-Answer'] = llm_responses_df['Filename'].map(answers_dict)
except:
    llm_responses_df = pd.read_csv('LLM_responses.csv')
    try:
        llm_responses_df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'],inplace=True, errors='ignore')
    except: pass
llm_responses_df.tail()

In [ ]:
llm_responses_df.to_csv('LLM_responses.csv')

In [ ]:
llm_responses_df.shape

# RAGAS

### Faithfulness (depends on context, doesn't on Ground Truth):

1. LLM = NA: NaN
2. LLM = Anything else: Ragas

### Answer Correctness (depends on Ground Truth, doesn't on Context):

1. Answer = NA, \
      LLM = NA: 1.0\
      LLM = Anything else: 0.0
3. LLM = No, \
      Answer = No: 1.0\
      Answer = Yes: 0.0
5. LLM = Yes, \
      Answer = No: 0.0\
      Answer = Yes: 1.0
7. LLM = Anything else: Ragas

In [ ]:
llm_responses_df = pd.read_csv('LLM_responses.csv')
try:
    llm_responses_df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'],inplace=True, errors='ignore')
except:
    pass
llm_responses_df.head()

In [ ]:
def clean_text(text):
    return text.strip().strip('.?!').lower()

In [ ]:
def clean_chunk(text: str) -> str:
    # Remove lines that are just artifacts like "Execution  Copy"
    # text = re.sub(r'^\s*(Execution\s+Copy)\s*$', '', text, flags=re.MULTILINE)

    # Replace multiple spaces with a single space
    text = re.sub(r' {2,}', ' ', text)

    # Replace multiple newlines with a single newline
    text = re.sub(r'\n{2,}', '\n', text)

    # Strip leading/trailing whitespace from each line
    text = '\n'.join(line.strip() for line in text.splitlines())

    # Strip leading/trailing whitespace from the whole text
    text = text.strip()

    return text

In [ ]:
chat_client = AsyncOpenAI(
    api_key=os.environ['OPENROUTER_API_KEY'],
    base_url="https://openrouter.ai/api/v1",
)
llm = llm_factory(llm_judge, provider="openai", client=chat_client)

In [ ]:
def calculate_ragas(user_query, llm_response, retrieved_chunks, answer = '', score = 'faithfulness'):
    metric_mapping = {
        'faithfulness': [Faithfulness(llm = llm)],
        'answer_correctness': [AnswerCorrectness(llm = llm)],
        'Both': [Faithfulness(llm = llm), AnswerCorrectness(llm = llm)]
    }

    data = {
            "question": [user_query],
            "answer": [llm_response],
            "contexts": [retrieved_chunks],
            "ground_truth": [answer]
        }

    dataset = Dataset.from_dict(data)

    eval_args = {
        "dataset": dataset,
        "metrics": metric_mapping[score],
        "raise_exceptions": True,
        "embeddings": free_embeddings,
        'llm': llm
    }

    result = evaluate(**eval_args)

    return result

In [ ]:
def ragas_logic(user_query, llm_response, retrieved_chunks, answer):
    result = {}
    if answer != 'No' and not pd.isna(answer):
        if clean_text(llm_response) == 'na':
            result['faithfulness'] = np.nan
            result['answer_correctness'] = 0.0
        elif clean_text(llm_response) == 'no':
            # calculate faithfulness
            faithfulness = calculate_ragas(user_query, llm_response, retrieved_chunks)
            result['faithfulness'] = faithfulness['faithfulness'][0]
            result['answer_correctness'] = 0.0
        elif clean_text(llm_response) == 'yes':
            # calculate faithfulness
            faithfulness = calculate_ragas(user_query, llm_response, retrieved_chunks)
            result['faithfulness'] = faithfulness['faithfulness'][0]
            result['answer_correctness'] = 1.0
        else:
            faithfulness_ans_corr = calculate_ragas(user_query, llm_response, retrieved_chunks, answer, 'Both')
            result['faithfulness'] = faithfulness_ans_corr['faithfulness'][0]
            result['answer_correctness'] = faithfulness_ans_corr['answer_correctness'][0]
    elif pd.isna(answer):
        if clean_text(llm_response) == 'na':
            result['faithfulness'] = np.nan
            result['answer_correctness'] = 1.0
        else:
            faithfulness = calculate_ragas(user_query, llm_response, retrieved_chunks)
            result['faithfulness'] = faithfulness['faithfulness'][0]
            result['answer_correctness'] = 0.0
    elif answer == "No":
        if clean_text(llm_response) == 'na':
            result['faithfulness'] = np.nan
            result['answer_correctness'] = 1.0
        elif clean_text(llm_response) == 'no':
            faithfulness = calculate_ragas(user_query, llm_response, retrieved_chunks)
            result['faithfulness'] = faithfulness['faithfulness'][0]
            result['answer_correctness'] = 1.0
        else:
            faithfulness = calculate_ragas(user_query, llm_response, retrieved_chunks)
            result['faithfulness'] = faithfulness['faithfulness'][0]
            result['answer_correctness'] = 0.0

    return result

In [ ]:
ragas_columns = [col.removesuffix('-faithfulness') for col in ragas_scores_df.columns if col.endswith("-faithfulness")]
ragas_columns

In [ ]:
column_query = {}
for col_collection in query_collection.get(include=['metadatas'])['metadatas']:
    source = col_collection['source']
    if source not in ragas_columns:
        col_query = query_collection.get(where = {'source': source}, include = ['documents'])
        col_query = col_query['documents'][0]
        column_query[source] = col_query

In [ ]:
user_query = list(column_query.values())[0]

retrieved_column, template_chunk, template_embedding = template_chunk_retrieval(user_query, query_collection)

print('Column:', retrieved_column)
print('Template chunk:', template_chunk)

In [ ]:
ragas_scores = {}
faithfulness_scores = {}
answer_correctness_scores = {}

In [ ]:
def ragas_loop(retrieved_column, template_chunk, template_embedding):
    ragas_scores = {}
    faithfulness_scores = {}
    answer_correctness_scores = {}
    for pdf_file in pdf_files_paths[:200]:
        pdf_filename = pdf_file.name
        name = Path(pdf_file).stem.replace(" ", "-")

        row = llm_responses_df[llm_responses_df['Filename'] == pdf_filename]
        llm_response = row[f'{retrieved_column}-LLM'].values[0]
        if pd.isna(llm_response):
            llm_response = 'na'
        retrieved_chunks = hybrid_search(template_chunk, template_embedding[0], pdf_collection, name, use_cross_enc = False, top_k = 1)
        retrieved_chunks = [clean_chunk(chunk) for chunk in retrieved_chunks]
        answer = row[f'{retrieved_column}-Answer'].values[0]

        result = ragas_logic(user_query, llm_response, retrieved_chunks, answer)
        ragas_scores[pdf_filename] = result
        print('done with file:', pdf_filename)

        faithfulness_scores[pdf_filename] = ragas_scores[pdf_filename]['faithfulness']
        answer_correctness_scores[pdf_filename] = ragas_scores[pdf_filename]['answer_correctness']

    return ragas_scores, faithfulness_scores, answer_correctness_scores

In [ ]:
for user_query in list(column_query.values())[5:8]:
    retrieved_column, template_chunk, template_embedding = template_chunk_retrieval(user_query, query_collection)

    ragas_scores, faithfulness_scores, answer_correctness_scores = ragas_loop(retrieved_column, template_chunk, template_embedding)

    ragas_scores_df[f'{retrieved_column}-faithfulness'] = ragas_scores_df['Filename'].map(faithfulness_scores)
    ragas_scores_df[f'{retrieved_column}-answer correctness'] = ragas_scores_df['Filename'].map(answer_correctness_scores)
    ragas_scores_df.to_csv(f'ragas_scores.csv')

In [ ]:
print(retrieved_column)
print(len(ragas_scores.keys()))
ragas_scores

In [ ]:
ragas_scores_df.tail()

In [ ]:
for pdf_file in pdf_files_paths[:200]:
    pdf_filename = pdf_file.name

    faithfulness_scores[pdf_filename] = ragas_scores[pdf_filename]['faithfulness']
    answer_correctness_scores[pdf_filename] = ragas_scores[pdf_filename]['answer_correctness']

    # print('done with file:', pdf_filename)

In [ ]:
# ragas_scores_df = pd.DataFrame({
#     f'{retrieved_column}-faithfulness': faithfulness_scores,
#     f'{retrieved_column}-answer correctness': answer_correctness_scores
# })
# ragas_scores_df = ragas_scores_df.rename_axis('Filename').reset_index()
# ragas_scores_df.tail()

In [ ]:
ragas_scores_df[f'{retrieved_column}-faithfulness'] = ragas_scores_df['Filename'].map(faithfulness_scores)
ragas_scores_df[f'{retrieved_column}-answer correctness'] = ragas_scores_df['Filename'].map(answer_correctness_scores)
ragas_scores_df.tail()

In [ ]:
ragas_scores_df.to_csv(f'ragas_scores.csv')

In [ ]:
print('Mean Faithfulness', np.nanmean(list(faithfulness_scores.values())))
print('Mean Answer Correctness', np.mean(list(answer_correctness_scores.values())))

In [ ]:
ragas_scores_df = pd.read_csv('ragas_scores.csv')
try:
    ragas_scores_df.drop(columns=['Unnamed: 0'],inplace=True, errors='ignore')
except:
    pass

# REST API

In [ ]:
import logging
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

In [ ]:
# ── Build FAISS index from chunks ─────────────────────────────────────────────
def build_faiss_index(chunks: list[str]) -> tuple:
    embeddings = embed_chunks(chunks, embedding_model)
    embeddings_np = np.array(embeddings)

    dimension = embeddings_np.shape[1]                           # e.g. 1024 for bge-large
    index     = faiss.IndexFlatIP(dimension)                     # Inner Product = cosine sim (for normalized embeddings)
    index.add(embeddings_np)

    return index, embeddings_np


def hybrid_search_faiss(query, query_embedding, chunks, index, use_cross_enc=True, top_k=5):

    # 1. Dense retrieval — FAISS instead of ChromaDB
    query_np      = np.array([query_embedding]).astype("float32")
    distances, indices = index.search(query_np, k=10)            # top 10 nearest neighbors
    dense_ids     = [f"chunk_{i}" for i in indices[0]]     # match your existing ID format

    # 2. Sparse retrieval (BM25) — unchanged
    tokenized_chunks = [tokenizer.tokenize(chunk) for chunk in chunks]
    bm25             = BM25Okapi(tokenized_chunks)
    tokenized_query  = tokenizer.tokenize(query)
    bm25_scores      = bm25.get_scores(tokenized_query)
    top_sparse_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True)[:20]
    sparse_ids = [f"chunk_{i}" for i in top_sparse_indices]

    # 3. RRF merge — unchanged
    def reciprocal_rank_fusion(dense_ids, sparse_ids, k=60):
        scores = {}
        for rank, doc_id in enumerate(dense_ids):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        for rank, doc_id in enumerate(sparse_ids):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)

    # Helper — convert chunk ID back to text
    def ids_to_texts(ids: list[str]) -> list[str]:
        return [
            chunks[int(id.replace(f"chunk_", ""))]
            for id in ids
            if int(id.replace(f"chunk_", "")) < len(chunks)
        ]

    if use_cross_enc:
        candidate_ids   = reciprocal_rank_fusion(dense_ids, sparse_ids)[:10]
        candidate_texts = ids_to_texts(candidate_ids)

        # 4. Cross-encoder reranking — unchanged
        rerank_pairs    = [[query, text] for text in candidate_texts]
        scores          = cross_encoder_model.predict(rerank_pairs)

        doc_score_pairs = sorted(zip(candidate_texts, scores), key=lambda x: x[1], reverse=True)
        return [text for text, score in doc_score_pairs[:top_k]]

    else:
        final_ids   = reciprocal_rank_fusion(dense_ids, sparse_ids)[:top_k]
        final_texts = ids_to_texts(final_ids)
        return final_texts

In [ ]:
file_path = f'{cwd}/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf'

In [ ]:
def rag(pdf, query):
    text = extract_text_from_pdf(pdf)
    paragraphs = text_splitter.split(text, do_paragraph_segmentation=True)
    chunks = extract_chunks(paragraphs)
    index, _        = build_faiss_index(chunks)
    print('done with vectorizing')

    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]

    retrieved_chunks = hybrid_search_faiss(
    query          = query,
    query_embedding= query_embedding,
    chunks         = chunks,
    index          = index,
    use_cross_enc  = True,
    top_k          = 1
    )
    print('done with retrieval')

    messages = build_prompt(
        query          = query,
        chunks         = retrieved_chunks,
    )

    outputs = pipe(
    messages
    )

    print('done with generation')

    response = outputs[0]["generated_text"][-1]["content"]
    return response

In [ ]:
query

In [ ]:
response = rag(file_path, query)
response

In [ ]:
app    = FastAPI()
logger = logging.getLogger(__name__)

In [ ]:
class QueryRequest(BaseModel):
    text:  str
    query: str

@app.post("/query")
def query_rag(request: QueryRequest):
    try:
        answer = rag(request.text, request.query)
        return {"answer": answer}
    except Exception as e:
        logger.error(f"Query failed: {str(e)}")
        raise

In [ ]:
@app.get("/health")
def health():
    return {"status": "ok"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Appendix

## Naive Retrieval

In [ ]:
def retrieve(query, model, index, chunks, embeddings, top_k=5):
    query_vec = model.encode([query], convert_to_numpy=True)
    D, I = index.search(query_vec, top_k)
    return [chunks[i] for i in I[0]], I[0]

In [ ]:
relevant_chunks, chunk_indices = retrieve(query, model, index, chunks, embeddings)

In [ ]:
for i, chunk in enumerate(relevant_chunks):
    print(f"Chunk {i+1}: {chunk}\n")

In [ ]:
client = chromadb.PersistentClient(path="./cuad_vector_db")

In [ ]:
for pdf_file in pdf_files_paths[2:60]:
    pdf_filename = pdf_file.name
    name = Path(pdf_file).stem.replace(" ", "-")
    collection = client.get_collection(name=name)
    vector_data = collection.get(include=['documents', 'embeddings'])

    ids = [name+id for id in vector_data['ids']]
    metadatas = [{'source': name}]*len(vector_data['documents'])

    new_collection.add(ids = ids,
                       documents = vector_data['documents'],
                       embeddings = vector_data['embeddings'],
                       metadatas = metadatas)
    assert vector_data['embeddings'].shape == new_collection.get(where = {'source': name}, include= ['embeddings'])['embeddings'].shape

    print(f'Finished with {name}')

In [ ]:
for file_path in pdf_files_paths[50:60]:
    # Create a collection (similar to a table in SQL)
    name = Path(file_path).stem.replace(" ", "-") # Replace spaces with hyphens for valid collection name
    collection = client.get_or_create_collection(name=name)

    text = extract_text_from_pdf(file_path)
    print('extracted text')
    paragraphs = sat.split(text, do_paragraph_segmentation=True)
    print('extracted paragraph')

    chunks = extract_chunks(paragraphs)
    embeddings = embed_chunks(chunks, embedding_model)
    print('extracted embeddings')

    collection.add(
      documents = chunks,
      embeddings = embeddings,
      ids=[f"chunk_{i}" for i in range(len(chunks))]
    )

    print(f'Finished with {name}')

### Editing old ids

In [ ]:
pdf_file = pdf_files_paths[0]
pdf_filename = pdf_file.name
name = Path(pdf_file).stem.replace(" ", "-")
name

In [ ]:
collection = client.get_collection(name=name)
existing_item = collection.get(include=["embeddings", "metadatas", "documents"])

In [ ]:
ids = [name+id for id in existing_item['ids']]
metadatas = [{'source': name}]*len(existing_item['documents'])

In [ ]:
new_collection.add(ids = ids,
                    documents = existing_item['documents'],
                    embeddings = existing_item['embeddings'],
                    metadatas = metadatas)

In [ ]:
new_collection.delete(ids=existing_item['ids'])


In [ ]:
answer_cols = [col for col in master_clauses.columns if col.endswith("-Answer")]
context_cols = [col for col in master_clauses.columns if not col.endswith("-Answer")]

In [ ]:
from collections import Counter

In [ ]:
string_cols = ['Document Name-Answer', 'Parties-Answer', 'Agreement Date-Answer', 'Effective Date-Answer','Expiration Date-Answer','Renewal Term-Answer','Notice Period To Terminate Renewal-Answer','Governing Law-Answer']

In [ ]:
for col in context_cols:
    try:
        if col+'-Answer' not in string_cols and col != 'Filename':
            contexts = master_clauses[col].values
            answers = master_clauses[col+'-Answer'].values
            empty_contexts = sum(1 for item in contexts if item == '[]')
            no_answers = sum(1 for item in answers if item == 'No')
            yes_answers = sum(1 for item in answers if item == 'Yes')
            assert empty_contexts == no_answers
            assert no_answers + yes_answers == answers.shape[0]
    except:
        print(col)

In [ ]:
for col in string_cols:
    print(col, master_clauses[col].isna().sum())

In [ ]:
for col in answer_cols:
    if col not in string_cols:
        answers = master_clauses[col].values
        print(col)
        print(Counter(answers))
        print('\n')